# Highway Multi-Agent — Example Notebook

Demonstrates:
1. Constructing the multi-agent highway environment.
2. Random rollout with frame rendering.
3. IQL training (independent Q-learning per vehicle).
4. Reward curve plot.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from discrete_action_space.highway_marl import make_pz_env
from discrete_action_space.marl_utils import random_rollout

## 1. Construct the environment

In [ ]:
env = make_pz_env(n_agents=2, vehicles_count=15, render_mode='rgb_array')

print('Agents       :', env.possible_agents)
print('Obs space    :', env.observation_space(env.possible_agents[0]))
print('Action space :', env.action_space(env.possible_agents[0]))

## 2. Random rollout with rendering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

obs, _ = env.reset()
frames = []
while env.agents:
    actions = {a: env.action_space(a).sample() for a in env.agents}
    obs, rewards, terms, truncs, infos = env.step(actions)
    frame = env.render()
    if frame is not None:
        frames.append(frame)

print(f'Episode done: {len(frames)} frames rendered')

# Show a few frames
if frames:
    fig, axes = plt.subplots(1, min(3, len(frames)), figsize=(12, 4))
    if min(3, len(frames)) == 1:
        axes = [axes]
    for ax, frame in zip(axes, frames[::max(1, len(frames) // 3)]):
        ax.imshow(frame)
        ax.axis('off')
    plt.suptitle('Highway MARL — random rollout frames')
    plt.tight_layout()
    plt.show()

env.close()

## 3. Baseline Training

In [ ]:
print('The old hand-written training helper has been removed. Use a dedicated baseline framework for training.')

## 4. Reward curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ep_rewards = results['episode_rewards']
fig, ax = plt.subplots(figsize=(10, 4))
for agent, rews in ep_rewards.items():
    window = 30
    rolling = np.convolve(rews, np.ones(window) / window, mode='valid')
    ax.plot(rolling, label=agent)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Reward (rolling avg {window})')
ax.set_title('Highway MARL — IQL training')
ax.legend()
plt.tight_layout()
plt.savefig('checkpoints/highway_iql/training_curve.png', dpi=120)
plt.show()